# 08: Compression Metrics Analysis (BPB & Perplexity)

This notebook analyzes the **compression efficiency** and **predictive quality** of H-Net models on chemical SMILES data.

## Metrics from the H-Net Paper

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Bits-Per-Byte (BPB)** | `CE_loss / ln(2)` | How many bits needed per byte? Lower = better compression |
| **Perplexity (PPL)** | `exp(CE_loss)` | How "surprised" is the model? Lower = more confident |

## Reference Points
- Random prediction (byte-level): **8.0 BPB**, **256 PPL**
- Well-trained English LM: ~1.0-1.5 BPB
- H-Net on DNA: Nearly 4× improvement over baselines

## Research Questions
- **F.1**: How do BPB/PPL improve with more training?
- **F.2**: Do polymers and molecules compress differently?
- **F.3**: Do 2-stage models achieve better compression?
- **F.4**: Does concatenation improve compression?
- **F.5**: How does H-Net compare to SmilesPE (proxy)?


In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

# Setup plotting style
sns.set_theme(style='whitegrid', context='talk', palette='mako')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

# Project paths
project_root = Path('/home/ec2-user/hnet_smiles')
analysis_dir = project_root / 'analysis'
checkpoints_dir = project_root / 'checkpoints'
figures_dir = analysis_dir / 'figures'
data_dir = analysis_dir / 'data'

figures_dir.mkdir(parents=True, exist_ok=True)
print(f"Figures will be saved to: {figures_dir}")


In [ ]:
# Core Functions: BPB and Perplexity Calculations

def ce_to_bpb(ce_loss: float) -> float:
    """Convert Cross-Entropy loss (nats) to Bits-Per-Byte."""
    return ce_loss / np.log(2)

def ce_to_perplexity(ce_loss: float) -> float:
    """Convert Cross-Entropy loss (nats) to Perplexity."""
    return np.exp(ce_loss)

def load_training_history(model_path: Path) -> dict:
    """Load training history from metadata.json."""
    metadata_path = model_path / 'metadata.json'
    if not metadata_path.exists():
        raise FileNotFoundError(f"No metadata.json found at {metadata_path}")
    with open(metadata_path, 'r') as f:
        return json.load(f)

def extract_compression_metrics(model_path: Path) -> pd.DataFrame:
    """Extract BPB and perplexity from training history."""
    metadata = load_training_history(model_path)
    history = metadata.get('training_history', [])
    
    records = []
    for entry in history:
        metrics = entry.get('metrics', {})
        ce_loss = metrics.get('ce_loss')
        if ce_loss is not None:
            records.append({
                'training_bytes': entry.get('cumulative_training_bytes', 0),
                'ce_loss': ce_loss,
                'bpb': ce_to_bpb(ce_loss),
                'perplexity': ce_to_perplexity(ce_loss),
                'checkpoint_type': entry.get('checkpoint_type', 'unknown'),
            })
    return pd.DataFrame(records)

def get_final_metrics(model_path: Path) -> dict:
    """Get final compression metrics for a model."""
    df = extract_compression_metrics(model_path)
    if df.empty:
        return None
    final = df.loc[df['training_bytes'].idxmax()]
    best_ce = df['ce_loss'].min()
    return {
        'final_bpb': final['bpb'],
        'final_perplexity': final['perplexity'],
        'best_bpb': ce_to_bpb(best_ce),
        'total_training_bytes': final['training_bytes'],
    }

print("Functions defined ✓")


In [ ]:
# Define all models to analyze
MODELS = {
    # 1-Stage Architecture
    'PI1M_noconcat_5epoch': {
        'path': checkpoints_dir / 'run_large_20251111_075600',
        'dataset': 'PI1M', 'concatenate': False, 'epochs': 5, 'architecture': '1-stage',
    },
    'PI1M_concat_1epoch': {
        'path': checkpoints_dir / 'run_large_20251113_181705',
        'dataset': 'PI1M', 'concatenate': True, 'epochs': 1, 'architecture': '1-stage',
    },
    'PI1M_concat_5epoch': {
        'path': checkpoints_dir / 'run_large_20251111_181836',
        'dataset': 'PI1M', 'concatenate': True, 'epochs': 5, 'architecture': '1-stage',
    },
    'PI1M_concat_22epoch': {
        'path': checkpoints_dir / 'run_large_20251112_150502',
        'dataset': 'PI1M', 'concatenate': True, 'epochs': 22, 'architecture': '1-stage',
    },
    'MOSES_noconcat_5epoch': {
        'path': checkpoints_dir / 'run_large_20251113_074900',
        'dataset': 'MOSES', 'concatenate': False, 'epochs': 5, 'architecture': '1-stage',
    },
    'MOSES_concat_5epoch': {
        'path': checkpoints_dir / 'run_large_20251112_071557',
        'dataset': 'MOSES', 'concatenate': True, 'epochs': 5, 'architecture': '1-stage',
    },
    # 2-Stage Architecture (Training Complete)
    'PI1M_concat_5epoch_2stage': {
        'path': checkpoints_dir / 'run_large_20260115_191350',
        'dataset': 'PI1M', 'concatenate': True, 'epochs': 5, 'architecture': '2-stage',
    },
    'MOSES_concat_5epoch_2stage': {
        'path': checkpoints_dir / 'run_large_20260116_074355',
        'dataset': 'MOSES', 'concatenate': True, 'epochs': 5, 'architecture': '2-stage',
    },
}

print(f"Defined {len(MODELS)} models to analyze")


## F.1: Load and Analyze All Models


In [ ]:
# Load compression metrics for all models
all_metrics = {}
all_histories = {}

for model_name, model_info in MODELS.items():
    model_path = model_info['path']
    
    if not model_path.exists():
        print(f"⏳ {model_name}: Path not found (training may be pending)")
        all_metrics[model_name] = None
        continue
    
    try:
        metrics = get_final_metrics(model_path)
        if metrics:
            metrics.update(model_info)
            all_metrics[model_name] = metrics
            all_histories[model_name] = extract_compression_metrics(model_path)
            print(f"✓ {model_name}: BPB={metrics['final_bpb']:.3f}, PPL={metrics['final_perplexity']:.2f}")
        else:
            print(f"⚠ {model_name}: No metrics found")
            all_metrics[model_name] = None
    except Exception as e:
        print(f"✗ {model_name}: Error - {e}")
        all_metrics[model_name] = None

print(f"\nLoaded {sum(1 for v in all_metrics.values() if v is not None)} models successfully.")


In [ ]:
# Create summary table
summary_data = []
for model_name, metrics in all_metrics.items():
    info = MODELS[model_name]
    if metrics is None:
        summary_data.append({
            'Model': model_name, 'Dataset': info['dataset'],
            'Concat': 'Yes' if info['concatenate'] else 'No',
            'Epochs': info['epochs'], 'Arch': info['architecture'],
            'Final BPB': '⏳', 'Final PPL': '⏳', 'Training Bytes': '⏳',
        })
    else:
        summary_data.append({
            'Model': model_name, 'Dataset': metrics['dataset'],
            'Concat': 'Yes' if metrics['concatenate'] else 'No',
            'Epochs': metrics['epochs'], 'Arch': metrics['architecture'],
            'Final BPB': f"{metrics['final_bpb']:.3f}",
            'Final PPL': f"{metrics['final_perplexity']:.2f}",
            'Training Bytes': f"{metrics['total_training_bytes']/1e6:.1f}M",
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)


## F.2: Training Dynamics - BPB Over Training Bytes


In [ ]:
# Plot BPB vs Training Bytes for all models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for model_name, history_df in all_histories.items():
    if history_df is None or history_df.empty:
        continue
    
    model_info = MODELS[model_name]
    ax = axes[0] if model_info['dataset'] == 'PI1M' else axes[1]
    
    linestyle = '--' if model_info['architecture'] == '2-stage' else '-'
    ax.plot(
        history_df['training_bytes'] / 1e6, 
        history_df['bpb'],
        label=model_name.replace('_', ' '),
        linestyle=linestyle,
        linewidth=2,
    )

for ax, title in zip(axes, ['PI1M (Polymer)', 'MOSES (Molecular)']):
    ax.set_xlabel('Training Bytes (M)')
    ax.set_ylabel('Bits-Per-Byte (BPB)')
    ax.set_title(f'{title} - Compression Efficiency')
    ax.legend(loc='upper right', fontsize=8)
    ax.axhline(y=8, color='red', linestyle=':', alpha=0.3, label='Random (8 BPB)')
    ax.set_ylim(bottom=0)

plt.tight_layout()
plt.savefig(figures_dir / 'compression_bpb_training_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {figures_dir / 'compression_bpb_training_dynamics.png'}")


## F.3: Final BPB Comparison Bar Chart


In [ ]:
# Create comparison bar chart
from matplotlib.patches import Patch

available_metrics = {k: v for k, v in all_metrics.items() if v is not None}

if available_metrics:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    model_names = list(available_metrics.keys())
    bpb_values = [m['final_bpb'] for m in available_metrics.values()]
    architectures = [MODELS[k]['architecture'] for k in model_names]
    datasets = [MODELS[k]['dataset'] for k in model_names]
    
    # Color by dataset, pattern by architecture
    colors = ['#4e79a7' if d == 'PI1M' else '#59a14f' for d in datasets]
    hatches = ['/' if a == '2-stage' else '' for a in architectures]
    
    bars = ax.bar(range(len(model_names)), bpb_values, color=colors)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
    
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([n.replace('_', '\n') for n in model_names], rotation=45, ha='right')
    ax.set_ylabel('Bits-Per-Byte (BPB)')
    ax.set_title('Final Compression Efficiency by Model (Lower is Better)')
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, bpb_values)):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Legend
    legend_elements = [
        Patch(facecolor='#4e79a7', label='PI1M (Polymer)'),
        Patch(facecolor='#59a14f', label='MOSES (Molecular)'),
        Patch(facecolor='gray', hatch='/', label='2-Stage Architecture'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    ax.axhline(y=8, color='red', linestyle=':', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(figures_dir / 'compression_bpb_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {figures_dir / 'compression_bpb_comparison.png'}")
else:
    print("No models with metrics available yet.")


## F.4: Architecture Comparison (1-Stage vs. 2-Stage)


In [ ]:
# Compare 1-stage vs 2-stage under same conditions
arch_comparisons = [
    ('PI1M_concat_5epoch', 'PI1M_concat_5epoch_2stage'),
    ('MOSES_concat_5epoch', 'MOSES_concat_5epoch_2stage'),
]

print("=== Architecture Comparison: 1-Stage vs 2-Stage ===\n")
for one_stage, two_stage in arch_comparisons:
    m1 = all_metrics.get(one_stage)
    m2 = all_metrics.get(two_stage)
    dataset = MODELS[one_stage]['dataset']
    
    if m1 and m2:
        bpb_change = (m1['final_bpb'] - m2['final_bpb']) / m1['final_bpb'] * 100
        ppl_change = (m1['final_perplexity'] - m2['final_perplexity']) / m1['final_perplexity'] * 100
        
        print(f"{dataset}:")
        print(f"  1-Stage BPB: {m1['final_bpb']:.3f} → 2-Stage: {m2['final_bpb']:.3f} ({bpb_change:+.1f}%)")
        print(f"  1-Stage PPL: {m1['final_perplexity']:.2f} → 2-Stage: {m2['final_perplexity']:.2f} ({ppl_change:+.1f}%)")
        print()
    elif m1 and not m2:
        print(f"{dataset}:")
        print(f"  1-Stage: BPB={m1['final_bpb']:.3f}, PPL={m1['final_perplexity']:.2f}")
        print(f"  2-Stage: ⏳ Training pending")
        print()
    else:
        print(f"{dataset}: Data not available\n")


## F.5: SmilesPE Comparison (Compression Ratio Proxy)

**Important Note**: SmilesPE is a tokenizer, NOT a language model. We cannot directly compute BPB/PPL for it.

Instead, we compare **compression ratios** as a proxy:
- H-Net: End-to-end learned compression → actual BPB
- SmilesPE: Static tokenization → `bytes / tokens` as compression ratio

For a fair comparison, one would need to train an LM on SmilesPE tokens and compute its BPB.


In [ ]:
# Load SmilesPE statistics and compare compression ratios
stats_dir = data_dir / 'statistics'
smilesPE_stats = {}

for dataset in ['PI1M', 'MOSES']:
    stats_path = stats_dir / f'SmilesPE_{dataset}_statistics.json'
    if stats_path.exists():
        with open(stats_path, 'r') as f:
            smilesPE_stats[dataset] = json.load(f)

if smilesPE_stats:
    print("=== SmilesPE vs H-Net Compression Comparison ===\n")
    print("Note: SmilesPE BPB is a theoretical estimate based on vocab size and avg token length\n")
    
    for dataset, stats in smilesPE_stats.items():
        avg_token_len = stats.get('mean_token_length', 4.5)
        vocab_size = 30000  # Approximate SmilesPE vocab
        theoretical_bpb = np.log2(vocab_size) / avg_token_len
        
        # Find corresponding H-Net model
        hnet_key = f'{dataset}_concat_5epoch'
        hnet_metrics = all_metrics.get(hnet_key)
        
        print(f"{dataset}:")
        print(f"  SmilesPE: ~{avg_token_len:.1f} bytes/token → ~{theoretical_bpb:.2f} BPB (theoretical)")
        if hnet_metrics:
            print(f"  H-Net:    {hnet_metrics['final_bpb']:.3f} BPB (actual)")
            improvement = (theoretical_bpb - hnet_metrics['final_bpb']) / theoretical_bpb * 100
            print(f"  Difference: {improvement:+.1f}% (negative = H-Net worse)")
        else:
            print(f"  H-Net:    ⏳ Pending")
        print()
else:
    print("SmilesPE statistics not found. Run data generation first.")


## Summary: Why Compression Metrics Matter for Chemistry

### Key Insights

1. **BPB measures chemical "grammar" learning**
   - Lower BPB = model learned meaningful patterns (functional groups, rings, etc.)
   - SMILES has inherent structure that can be compressed

2. **Comparison to other domains**
   - English text: ~1.0-1.5 BPB
   - DNA with H-Net: ~4× improvement over baselines
   - SMILES: Expected to be in between (more structured than text)

3. **SmilesPE comparison caveats**
   - SmilesPE is a tokenizer, not an LM
   - For fair comparison, need to train LM on SmilesPE tokens
   - H-Net's advantage: End-to-end learned compression

### Potential Applications

- **Anomaly detection**: High PPL = unusual molecule
- **Transfer learning**: Better compression = better representations
- **Efficiency**: Lower BPB = more efficient encoding for downstream tasks


In [ ]:
# Save results
summary_df.to_csv(data_dir / 'compression_metrics_summary.csv', index=False)
print(f"Summary saved to: {data_dir / 'compression_metrics_summary.csv'}")

# Print best model
valid = {k: v for k, v in all_metrics.items() if v is not None}
if valid:
    best = min(valid.items(), key=lambda x: x[1]['final_bpb'])
    print(f"\n🏆 Best Compression: {best[0]}")
    print(f"   BPB: {best[1]['final_bpb']:.3f}")
    print(f"   PPL: {best[1]['final_perplexity']:.2f}")
